#### Importing Essentials

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timezone
from pyspark.sql import Row
import uuid 

#### Choose Catalog

In [0]:
spark.sql("use catalog novacart_adb")

#### Create Schema

In [0]:
spark.sql("create schema if not exists broze_schema")

# Step 2 - Bronze Control Table

This table stores the watermark for each source table.

It helps the pipeline remember:
  - the latest timestamp already processed.
  - the latest primary key processed at that timestamp
  - how many rows were written in the latest run

This is what makes the bronze load incremental and rerun-safe.

In [0]:
spark.sql("""
            create table if not exists novacart_adb.broze_schema.ingestion_control(
              layer string,
              table_name string,
              ts_col string, --timeStamp column
              pk_col string,
              last_successful_ts timestamp,
              last_successful_pk bigint,
              last_run_id string,
              rows_written bigint,
              run_status string,
              updated_at timestamp
            )
            using delta
          """

)

# Step 3 - Source Table Configuration
This cell defines which source tables will be loaded into bronze and which columns should be used as:
  - Primary Key
  - timestamp / watermark coloumn

It also creates a unique bronze_run_id for the current pipeline run.

In [0]:
tables_config = {
  "orders" : {"pk_col" : "order_id", "ts_col": "updated_at"},
  "products" : {"pk_col" : "product_id", "ts_col": "updated_at"},
  "payments" : {"pk_col" : "payment_id", "ts_col": "processed_at"}
}

bronze_run_id = str(uuid.uuid4())
print("Current Bronze Run ID:", bronze_run_id)

# Step 4 - Helper Functions
This cell contains reusable functions.
  - get_last_successfull_watermark() reads the last processed watermark from the control table
  - upsert_bronze_control() updates the control table after a successful Bronze load.

These functions keep the main function logic cleaner and easier to understand.

In [0]:
def get_last_successful_watermark(table_name:str):
    ctrl = (
        spark.table("novacart_adb.broze_schema.ingestion_control")
        .filter(
                (F.col("layer")=='bronze')&
                (F.col("table_name")==table_name)&
                (F.col("run_status")=='success')
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    rows = ctrl.collect()
    if not rows:
        return None, None

    return rows[0]["last_successful_ts"], rows[0]["last_successful_pk"]

In [0]:
def upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id):

    control_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.now(timezone.utc)   # ✅ fixed
        )],
        schema="""
        layer string,
        table_name string,
        ts_col string,
        pk_col string,
        last_successful_ts timestamp,
        last_successful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "novacart_adb.broze_schema.ingestion_control")  # ✅ fixed

    (dt.alias("t")
        .merge(
            control_df.alias("s"),   # ✅ FIXED
            "t.table_name = s.table_name AND t.layer = s.layer"
        )
        .whenMatchedUpdate(set={
            "ts_col": "s.ts_col",
            "pk_col": "s.pk_col",
            "last_successful_ts": "s.last_successful_ts",
            "last_successful_pk": "s.last_successful_pk",
            "last_run_id": "s.last_run_id",
            "rows_written": "s.rows_written",
            "run_status": "s.run_status",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

# Step 5 - Bronze Incremental Load Loop
This is main Bronze logic.

For each table the notebook:

    1.reads the last watermark
    2.reads the source SQL table
    3.filters only new / changed rows
    4.adds bronze audit columns
    5.appends the rows into Bronze Delta table
    6.updates the control table

This is the core incremental load logic.

In [0]:
for table_name, cfg in tables_config.items():
    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]

    source_table = f"`novacart_sql_connection_catalog`.dbo.{table_name}"
    target_table = f"novacart_adb.broze_schema.{table_name}_raw"

    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)

    print(f"\n== Processing {table_name} ===")
    print(f"Last successful ts : {last_successful_ts}")
    print(f"Last successful pk : {last_successful_pk}")

    source_df = spark.read.table(source_table)\
        .withColumn(ts_col, F.col(ts_col).cast("timestamp"))

    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful_ts)) |
            (
                (F.col(ts_col) == F.lit(last_successful_ts)) &
                (F.col(pk_col).cast("long") > F.lit(int(last_successful_pk)))
            )
        )

    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
        .withColumn("bronze_source_table", F.lit(source_table))
    )

    rows_count = rows_to_load.count()
    print(f"{table_name} rows_to_load = {rows_count}")

    if rows_count == 0:
        print(f"No New Rows for {table_name}.")
        upsert_bronze_control(
            table_name, ts_col, pk_col,
            last_successful_ts, last_successful_pk,
            rows_count, bronze_run_id
        )
        continue

    rows_to_load.write.format('delta').mode('append').saveAsTable(target_table)

    max_ts = rows_to_load.agg(F.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]

    max_pk = (
        rows_to_load
        .filter(F.col(ts_col) == F.lit(max_ts))
        .agg(F.max(F.col(pk_col).cast("long")).alias("max_pk"))
        .collect()[0]["max_pk"]
    )

    upsert_bronze_control(
        table_name, ts_col, pk_col,
        max_ts, max_pk,
        rows_count, bronze_run_id
    )

    print(f"Wrote {rows_count} rows to {target_table}")

# Step 6 - Quick Validation

This final cell prints the Bronze row counts and displays the control table so you can verify that the incremental logic is working correctly.

In [0]:
print("Orders Bronze Count :", spark.sql("select count(*) from novacart_adb.broze_schema.orders_raw").collect()[0][0])
print("Products Bronze Count :", spark.sql("select count(*) from novacart_adb.broze_schema.products_raw").collect()[0][0])
print("Payments Bronze Count :", spark.sql("select count(*) from novacart_adb.broze_schema.payments_raw").collect()[0][0])

display(spark.sql("select * from novacart_adb.broze_schema.ingestion_control").orderBy("table_name"))